# FixFirst AI - Colab Training Notebook

Use this notebook to train the Aspect Category and Aspect Sentiment models on Google Colab using a GPU. Colab offers free T4 GPUs which will dramatically speed up the fine-tuning process of the DistilBERT models compared to running them locally on a CPU.

## 1. Setup Environment

First, you need to upload your project codebase to Colab. The easiest way is to upload the entire `FixFirst_AI` folder to your Google Drive and mount it here. Alternatively, you can clone your GitHub repository if it's pushed online.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Change this path to where you uploaded your FixFirst_AI project in your Google Drive
# For example, if you uploaded it to the root of your Drive, it might be:
import os
os.chdir('/content/drive/MyDrive/FixFirst_AI')
!ls # Verify you are in the correct directory

## 2. Install Dependencies

Install the required packages for the pipeline and training (PyTorch, Transformers, MLflow, etc.).

In [ ]:
!pip install -r requirements.txt
!pip install -r requirements-training.txt

## 3. Data Preparation (Optional)

If you haven't already processed the data and extracted the gold labels locally, you can run the ingestion and extraction scripts here. 

*Note: Make sure your raw AWARE dataset is available at `data/raw/aware_reviews.csv`.*

In [ ]:
# Initialize SQLite database and seed features taxonomy
!PYTHONPATH=src python scripts/init_db.py
!PYTHONPATH=src python scripts/seed_features.py

# Ingest the AWARE CSV dataset into the database
!PYTHONPATH=src python scripts/ingest_aware.py

# Run preprocessing to split into train/val/test
!PYTHONPATH=src python scripts/run_preprocessing.py

# Extract the gold aspect and sentiment labels from the dataset into Parquet format for training
!PYTHONPATH=src python scripts/extract_gold_labels.py

## 4. Train Aspect Category Model

Train the multi-label aspect category classifier using the extracted labels. Make sure you have selected a GPU runtime (`Runtime > Change runtime type > Hardware accelerator > T4 GPU`).

In [ ]:
!PYTHONPATH=src python scripts/train_aspect_category.py

## 5. Train Aspect Sentiment Model

Train the sentiment classifier.

In [ ]:
!PYTHONPATH=src python scripts/train_aspect_sentiment.py

## 6. Evaluate Models

Run the evaluation script to see how the trained models perform on the held-out gold test set.

In [ ]:
!PYTHONPATH=src python scripts/run_gold_eval.py

## 7. Download Trained Models

After training, the models will be saved in `data/models/`. You can zip them and download them to your local machine to use for inference, or since you're connected to Google Drive, they should already be synced to your Drive!

In [ ]:
# If you are not using Google Drive mounting and just uploaded files to Colab directly:
# !zip -r models.zip data/models/
# from google.colab import files
# files.download('models.zip')